# Predicting Student Test Scores 
## Score: 9.26490

In [1]:
import time
import hashlib
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')

test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')

bin_src_cols = [c for c in ['study_hours', 'sleep_hours', 'class_attendance'] if c in X.columns]
for c in bin_src_cols:
    _, bins = pd.qcut(X[c], q=20, duplicates='drop', retbins=True)
    bins[0] = -np.inf
    bins[-1] = np.inf
    bc = f'bin_{c}'
    X[bc] = pd.cut(X[c], bins=bins, include_lowest=True).cat.codes.astype('int16')
    X_test[bc] = pd.cut(X_test[c], bins=bins, include_lowest=True).cat.codes.astype('int16')


In [3]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'n_estimators': 8000,
    'num_leaves': 79,
    'max_depth': 10,
    'min_child_samples': 55,
    'reg_alpha': 10.0,
    'reg_lambda': 0.50,
    'min_split_gain': 1e-6,
    'subsample': 0.72,
    'subsample_freq': 3,
    'colsample_bytree': 0.65,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds = [420, 666, 80085]
n_splits = 10

EARLY_STOP = 250
MAX_SECONDS = 3300

TE_SMOOTH = 25.0

EPS_Y = 1e-3

def y_to_z(y_arr):
    y_arr = np.asarray(y_arr, dtype=float)
    y_clip = np.clip(y_arr, EPS_Y, 100.0 - EPS_Y)
    p = y_clip / 100.0
    return np.log(p / (1.0 - p))

def z_to_y(z_arr):
    z_arr = np.asarray(z_arr, dtype=float)
    p = 1.0 / (1.0 + np.exp(-z_arr))
    return 100.0 * p

te_cols = [c for c in ['course', 'exam_difficulty', 'study_method', 'sleep_quality', 'facility_rating', 'internet_access', 'gender'] if c in X.columns]
te_cols += [c for c in X.columns if str(c).startswith('bin_')]
te_cols = list(dict.fromkeys(te_cols))

te_pairs = []
for a, b in [('course', 'exam_difficulty'), ('study_method', 'exam_difficulty'), ('course', 'study_method')]:
    if a in X.columns and b in X.columns:
        te_pairs.append((a, b))

for c in te_cols:
    vc = pd.concat([X[c], X_test[c]]).value_counts(dropna=False)
    X[f'ce_{c}'] = X[c].map(vc).astype(float).fillna(0.0)
    X_test[f'ce_{c}'] = X_test[c].map(vc).astype(float).fillna(0.0)

t0 = time.time()

alt_params = {
    **base_params,
    'num_leaves': 47,
    'max_depth': -1,
    'min_child_samples': 90,
    'reg_alpha': 20.0,
    'reg_lambda': 1.50,
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.85
}


y_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
y_bins = y_bins_cat.cat.codes.to_numpy()
min_count = int(pd.Series(y_bins).value_counts().min())
if n_splits > min_count:
    n_splits = max(2, min_count)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def te_fit_1(X_ref, y_ref, col, smooth):
    y_s = pd.Series(y_ref, index=X_ref.index)
    g = y_s.groupby(X_ref[col], observed=False).agg(['mean', 'count'])
    prior = float(y_s.mean())
    enc = (g['mean'] * g['count'] + prior * smooth) / (g['count'] + smooth)
    return enc, prior

def te_fit_2(X_ref, y_ref, a, b, smooth):
    y_s = pd.Series(y_ref, index=X_ref.index)
    key = X_ref[a].astype(str) + '|' + X_ref[b].astype(str)
    g = y_s.groupby(key).agg(['mean', 'count'])
    prior = float(y_s.mean())
    enc = (g['mean'] * g['count'] + prior * smooth) / (g['count'] + smooth)
    return enc, prior

def te_apply_1(X_df, col, enc, prior):
    return X_df[col].map(enc).astype(float).fillna(prior)

def te_apply_2(X_df, a, b, enc, prior):
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    return key.map(enc).astype(float).fillna(prior)

def add_te(X_tr, X_va, X_te, y_tr):
    add_tr = {}
    add_va = {}
    add_te2 = {}

    for c in te_cols:
        enc, prior = te_fit_1(X_tr, y_tr, c, TE_SMOOTH)
        name = f'te_{c}'
        add_tr[name] = te_apply_1(X_tr, c, enc, prior)
        add_va[name] = te_apply_1(X_va, c, enc, prior)
        add_te2[name] = te_apply_1(X_te, c, enc, prior)

    for a, b in te_pairs:
        enc, prior = te_fit_2(X_tr, y_tr, a, b, TE_SMOOTH)
        name = f'te_{a}__{b}'
        add_tr[name] = te_apply_2(X_tr, a, b, enc, prior)
        add_va[name] = te_apply_2(X_va, a, b, enc, prior)
        add_te2[name] = te_apply_2(X_te, a, b, enc, prior)

    X_tr2 = pd.concat([X_tr, pd.DataFrame(add_tr, index=X_tr.index)], axis=1)
    X_va2 = pd.concat([X_va, pd.DataFrame(add_va, index=X_va.index)], axis=1)
    X_te2 = pd.concat([X_te, pd.DataFrame(add_te2, index=X_te.index)], axis=1)

    return X_tr2, X_va2, X_te2

def cv_run(params_list, seeds, label):
    sum_oof = np.zeros(len(X), dtype=float)
    cnt_oof = np.zeros(len(X), dtype=float)
    sum_test = np.zeros(len(X_test), dtype=float)
    seeds_done = 0

    for s_i, seed in enumerate(seeds, start=1):
        params_list2 = params_list if isinstance(params_list, (list, tuple)) else [params_list]

        oof = np.full(len(X), np.nan, dtype=float)
        test_pred_sum = np.zeros(len(X_test), dtype=float)
        rmse_scores = []
        folds_done = 0

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_bins), start=1):
            if (time.time() - t0) > MAX_SECONDS:
                break

            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            y_tr_z = y_to_z(y_tr)
            y_va_z = y_to_z(y_va)

            X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr_z)

            va_preds = []
            te_preds = []
            rmses = []

            for params in params_list2:
                p = {**params, 'random_state': seed}
                model = lgb.LGBMRegressor(**p)
                model.fit(
                    X_tr2,
                    y_tr_z,
                    eval_set=[(X_va2, y_va_z)],
                    callbacks=[lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(0)]
                )

                va_p = z_to_y(model.predict(X_va2))
                te_p = z_to_y(model.predict(X_test2))
                rmse_p = float(np.sqrt(mean_squared_error(y_va, va_p)))

                va_preds.append(va_p)
                te_preds.append(te_p)
                rmses.append(rmse_p)

            inv = 1.0 / (np.square(np.array(rmses, dtype=float)) + 1e-12)
            w = inv / inv.sum()

            va_pred = np.zeros(len(va_idx), dtype=float)
            te_pred = np.zeros(len(X_test), dtype=float)
            for wi, va_p, te_p in zip(w, va_preds, te_preds):
                va_pred += wi * va_p
                te_pred += wi * te_p

            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f} | w: {w.tolist()} | rmse: {rmses}')

            test_pred_sum += te_pred
            folds_done += 1

        if folds_done == 0:
            break

        test_pred = test_pred_sum / folds_done

        filled = ~np.isnan(oof)
        oof_filled = np.clip(oof[filled], 0, 100)
        sum_oof[filled] += oof_filled
        cnt_oof[filled] += 1.0

        sum_test += np.clip(test_pred, 0, 100)
        seeds_done += 1

        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        if (time.time() - t0) > MAX_SECONDS:
            break

    denom = np.maximum(cnt_oof, 1.0)
    all_oof = np.clip(sum_oof / denom, 0, 100)

    if seeds_done > 0:
        all_test = np.clip(sum_test / seeds_done, 0, 100)
    else:
        all_test = np.zeros(len(X_test), dtype=float)

    filled_all = cnt_oof > 0
    if filled_all.any():
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all])))
    else:
        final_oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


lgb_oof, lgb_test, _ = cv_run([base_params, alt_params], seeds, 'LGB2')

pred = np.clip(lgb_test, 0, 100)

submission = pd.DataFrame({'id': test_ids, 'exam_score': pred})

out_path = 'submission.csv'
submission.to_csv(out_path, index=False)
with open(out_path, 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print(out_path)
print('md5', md5)
print('pred_mean', float(np.mean(pred)), 'pred_std', float(np.std(pred)), 'pred_min', float(np.min(pred)), 'pred_max', float(np.max(pred)))
print('pred_ge_99_5', int(np.sum(pred >= 99.5)), 'pred_ge_97', int(np.sum(pred >= 97.0)))


LGB2 SEED 420 (1/3)
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[523]	valid_0's rmse: 1.34116
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1146]	valid_0's rmse: 1.34026
  Fold 1/10 RMSE: 9.29615 | w: [0.5005519104773459, 0.499448089522654] | rmse: [9.301411319938573, 9.31168408543778]
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[598]	valid_0's rmse: 1.31292
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1051]	valid_0's rmse: 1.31099
  Fold 2/10 RMSE: 9.30145 | w: [0.5003162998310913, 0.4996837001689087] | rmse: [9.307918899983294, 9.313808949946129]
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[927]	valid_0's rmse: 1.3202
Training until validation scores don't improve for 250 rounds
Early stopping, best iteration is:
[1109]	vali